# Unión archivos CSV individuales por página de Google en un sólo CSV por país

In [ ]:
import pandas as pd
import glob
import os

# Ruta de la carpeta donde están los CSV
ruta_carpeta = "data-international-searches/links/links_usa" # Adecuar de acuerdo a cada país

# Ruta de salida dentro del proyecto
ruta_salida = os.path.join("data-international-searches/links", "links_usa.csv") # Adecuar de acuerdo a cada país

# Buscar todos los archivos CSV en la carpeta
archivos_csv = glob.glob(os.path.join(ruta_carpeta, "*.csv"))

# Lista para almacenar los DataFrames
dataframes = []

for archivo in archivos_csv:
    df = pd.read_csv(archivo)
    dataframes.append(df)

# Unir todos los DataFrames
df_final = pd.concat(dataframes, ignore_index=True)

# Eliminar duplicados si es necesario
df_final.drop_duplicates(inplace=True)

# Guardar en la ruta específica
df_final.to_csv(ruta_salida, index=False)

print(f"CSV unido guardado en: {ruta_salida}")


# Agregar columna "country" a cada CSV por país

In [ ]:
import pandas as pd
import os

# Configuración del script

# Nombre del archivo original (debe estar dentro de la carpeta 'data-international-searches/links')
nombre_archivo_original = "links_usa.csv"  # Adaptar esto por cada CSV

# Valor a asignar en la nueva columna 'country'
valor_country = "usa"  # Adaptar esto por cada CSV

# Rutas y operaciones

# Ruta del archivo original
ruta_entrada = os.path.join("data-international-searches/links", nombre_archivo_original)

# Cargar el CSV
df = pd.read_csv(ruta_entrada)

# Agregar la nueva columna
df["country"] = valor_country

# Crear nombre del nuevo archivo
nombre_base = os.path.splitext(nombre_archivo_original)[0]
nuevo_nombre_archivo = f"{nombre_base}_withcountrycolumn.csv"

# Ruta de la carpeta de salida
carpeta_salida = "data-international-searches/linkswithcountrycolumn"
os.makedirs(carpeta_salida, exist_ok=True)  # Crea la carpeta si no existe

# Ruta completa del nuevo archivo
ruta_salida = os.path.join(carpeta_salida, nuevo_nombre_archivo)

# Guardar el nuevo archivo CSV
df.to_csv(ruta_salida, index=False)

# Mensaje de confirmación
print(f"Nuevo archivo CSV guardado como: {ruta_salida}")


# Unir todos los CSV con columna "country" en un CSV único unificado

In [ ]:
import pandas as pd
import os

# Carpeta de entrada
carpeta_entrada = "data-international-searches/linkswithcountrycolumn"

# Carpeta de salida
carpeta_salida = "data-international-searches/linksall"
os.makedirs(carpeta_salida, exist_ok=True)

# Lista para almacenar DataFrames
dfs = []

# Recorrer todos los archivos CSV en la carpeta de entrada
for archivo in os.listdir(carpeta_entrada):
    if archivo.endswith(".csv"):
        ruta = os.path.join(carpeta_entrada, archivo)
        df = pd.read_csv(ruta)
        dfs.append(df)

# Unir todos los DataFrames
df_total = pd.concat(dfs, ignore_index=True)

# Guardar el resultado en la carpeta de salida
archivo_salida = os.path.join(carpeta_salida, "links_all.csv")
df_total.to_csv(archivo_salida, index=False)

# Confirmación
print(f"Archivo unido guardado como: {archivo_salida}")


# Agregar columna "httpcode" y obtener código http por cada link

In [ ]:
import pandas as pd
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm
import os

# Rutas
archivo_entrada = "data-international-searches/linksall/links_all.csv"
carpeta_salida = "data-international-searches/linksallwithhttpcode"
archivo_salida = os.path.join(carpeta_salida, "links_all_withhttpcode.csv")

# Crear carpeta si no existe
os.makedirs(carpeta_salida, exist_ok=True)

# Cargar el archivo original
df = pd.read_csv(archivo_entrada)

# Función para obtener el código HTTP
def obtener_http_code(url):
    try:
        response = requests.head(url, allow_redirects=True, timeout=5, headers={"User-Agent": "Mozilla/5.0"})
        return response.status_code
    except requests.RequestException:
        return None

# Lista de URLs
urls = df["Link"].tolist()

# Lista vacía para almacenar resultados
http_codes = [None] * len(urls)

# Proceso en paralelo
max_workers = 20  # Ajustable (10-50 es adecuado)
with ThreadPoolExecutor(max_workers=max_workers) as executor:
    future_to_index = {executor.submit(obtener_http_code, url): i for i, url in enumerate(urls)}

    for future in tqdm(as_completed(future_to_index), total=len(urls), desc="Obteniendo códigos HTTP"):
        i = future_to_index[future]
        try:
            http_codes[i] = future.result()
        except Exception:
            http_codes[i] = None

# Añadir la nueva columna
df["httpcode"] = http_codes

# Guardar el nuevo archivo CSV
df.to_csv(archivo_salida, index=False)

print(f"Proceso completado. Archivo guardado como: {archivo_salida}")


# Filtrar links por únicos y código HTTP 200

In [ ]:
import pandas as pd
import os

# Rutas de entrada y salida
archivo_entrada = "data-international-searches/linksallwithhttpcode/links_all_withhttpcode.csv"
carpeta_salida = "data-international-searches/linksallwithhttpcodeonlyunique200"
archivo_salida = os.path.join(carpeta_salida, "links_all_withhttpcode_onlyunique200.csv")

# Crear carpeta de salida si no existe
os.makedirs(carpeta_salida, exist_ok=True)

# Cargar el CSV
df = pd.read_csv(archivo_entrada)

# 1) Eliminar duplicados en la columna "Link"
df = df.drop_duplicates(subset="Link", keep="first")

# 2) Filtrar solo httpcode == 200 o 200.0
df = df[df["httpcode"].isin([200, 200.0])]

# Guardar el nuevo CSV
df.to_csv(archivo_salida, index=False)

print(f"Archivo filtrado y guardado en: {archivo_salida}")
print(f"Total de enlaces después del filtrado: {len(df)}")
